# OneVoice V2 — GIPFormer VI adaptation gate

Notebook này chỉ cho phép bước adaptation sau khi checkpoint PyTorch chính thức tải/giải mã tương thích với runtime ONNX. Upstream GIPFormer hiện công bố PyTorch inference nhưng không có construction-domain Icefall training recipe; notebook không giả vờ train khi recipe đó chưa được review. Cần Colab GPU Linux.

Mọi cache/model/report lưu trên Drive. Chỉ dùng split `dev`; không đọc `test`.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import json, os, subprocess, sys

GITHUB_REPO = 'https://github.com/Platypus27-coder/OneVoice.git'
REPO = Path('/content/OneVoice')
DRIVE_ROOT = Path('/content/drive/MyDrive/OneVoice')
MANIFEST = Path('/content/drive/MyDrive/onevoice_audio_v1/manifest.jsonl')
WORK_ROOT = DRIVE_ROOT / 'gipformer_vi_adaptation_v1'
MODEL_ROOT = DRIVE_ROOT / 'models/gipformer_pytorch_v1_official'
ICEFALL_ROOT = DRIVE_ROOT / 'model_cache/gipformer/icefall'
REPORT_ROOT = DRIVE_ROOT / 'reports/gipformer_vi_adaptation_v1'
for path in (WORK_ROOT, MODEL_ROOT, ICEFALL_ROOT.parent, REPORT_ROOT): path.mkdir(parents=True, exist_ok=True)
if (REPO / '.git').is_dir():
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only', 'origin', 'main'], check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', 'main', GITHUB_REPO, str(REPO)], check=True)
os.chdir(REPO); os.environ['PYTHONUNBUFFERED'] = '1'
if not MANIFEST.is_file(): raise FileNotFoundError(f'Missing VI manifest: {MANIFEST}')
gpu = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], text=True, capture_output=True)
if gpu.returncode: raise RuntimeError('GPU required: Colab Runtime → Change runtime type → T4/L4/A100, then Run all.')
print('GPU:', gpu.stdout.strip())


In [ ]:
# Reproducible source + model pins. `model.pt`, `bpe.model`, and `tokens.txt` are copied to Drive.
UPSTREAM_REPO = 'https://github.com/ggroup-ai-lab/gipformer.git'
UPSTREAM_COMMIT = 'c6abf2f244680a3be6ca4cd79c006dd32b8e6322'
UPSTREAM_DIR = Path('/content/gipformer-upstream')
HF_REPO = 'g-group-ai-lab/gipformer-65M-rnnt'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'uv', 'huggingface_hub'], check=True)
if not (UPSTREAM_DIR / '.git').is_dir(): subprocess.run(['git', 'clone', UPSTREAM_REPO, str(UPSTREAM_DIR)], check=True)
subprocess.run(['git', '-C', str(UPSTREAM_DIR), 'fetch', '--depth', '1', 'origin', UPSTREAM_COMMIT], check=True)
subprocess.run(['git', '-C', str(UPSTREAM_DIR), 'checkout', '--detach', UPSTREAM_COMMIT], check=True)
from huggingface_hub import HfApi, snapshot_download
HF_REVISION = HfApi().model_info(HF_REPO).sha
snapshot_download(HF_REPO, revision=HF_REVISION, local_dir=str(MODEL_ROOT), allow_patterns=['model.pt', 'bpe.model', 'tokens.txt', 'config.json'])
missing = [p for p in ('model.pt','bpe.model','tokens.txt') if not (MODEL_ROOT / p).is_file()]
if missing: raise FileNotFoundError(f'Missing official PyTorch artifacts: {missing}')
(WORK_ROOT / 'upstream_pin.json').write_text(json.dumps({'source_repo':UPSTREAM_REPO,'source_commit':UPSTREAM_COMMIT,'hf_repo':HF_REPO,'hf_revision':HF_REVISION}, indent=2), encoding='utf-8')
print('Pinned GIPFormer source/model:', UPSTREAM_COMMIT, HF_REVISION)


In [ ]:
# Isolated upstream CUDA stack. Drive FUSE cannot provide uv's file locks, so dependency cache stays local; model/report artifacts stay on Drive.
LOCAL_UV_CACHE = Path('/content/.cache/onevoice-uv')
LOCAL_UV_CACHE.mkdir(parents=True, exist_ok=True)
env = {**os.environ, 'UV_CACHE_DIR': str(LOCAL_UV_CACHE)}
command = ['uv', 'sync', '--extra', 'pytorch', '--verbose']
print('> ' + ' '.join(command), flush=True)
process = subprocess.Popen(command, cwd=UPSTREAM_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
assert process.stdout is not None
for line in process.stdout: print(line, end='', flush=True)
if process.wait(): raise RuntimeError('Upstream CUDA environment failed; the complete uv log is printed above. Do not continue.')
UPSTREAM_PYTHON = UPSTREAM_DIR / '.venv/bin/python'
probe = 'import json,torch,k2,kaldifeat; print(json.dumps({"torch":torch.__version__,"cuda":torch.cuda.is_available(),"k2":getattr(k2, "__version__", getattr(k2, "__dev_version__", "unknown"))}))'
check = subprocess.run([str(UPSTREAM_PYTHON), '-c', probe], text=True, capture_output=True)
print('GIPFormer environment exit code:', check.returncode)
print('stdout:', check.stdout, end='' if check.stdout.endswith('\n') else '\n')
print('stderr:', check.stderr, end='' if check.stderr.endswith('\n') else '\n')
if check.returncode: raise RuntimeError('Official PyTorch/Icefall imports failed; copy the stderr above. Do not continue.')
status = json.loads(check.stdout)
if not status.get('cuda'): raise RuntimeError(f'PyTorch cannot see the Colab GPU: {status}. Restart the GPU runtime once, then re-run this cell.')
print('CUDA-ready upstream environment:', status)


In [ ]:
# Compare current ONNX and official PyTorch on the same DEV/noisy slice. This cell resumes saved PyTorch batches.
import yaml
SMOKE_SAMPLES = 32
RUNTIME_ONNX = DRIVE_ROOT / 'models/gipformer'
if not RUNTIME_ONNX.is_dir(): raise FileNotFoundError(f'Missing runtime ONNX bundle: {RUNTIME_ONNX}')
config = yaml.safe_load((REPO / 'config/config.yaml').read_text(encoding='utf-8'))
config['asr']['gipformer_model_dir'] = str(RUNTIME_ONNX)
CONFIG = WORK_ROOT / 'benchmark_config.yaml'; CONFIG.write_text(yaml.safe_dump(config, allow_unicode=True, sort_keys=False), encoding='utf-8')

def run_streaming(command, label):
    print(f'[{label}] > ' + ' '.join(map(str, command)), flush=True)
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env={**os.environ, 'PYTHONUNBUFFERED':'1'})
    assert process.stdout is not None
    for line in process.stdout: print(line, end='', flush=True)
    code = process.wait()
    if code: raise RuntimeError(f'{label} failed with exit code {code}; complete traceback is printed above.')

ONNX_REPORT = REPORT_ROOT / 'onnx_dev_noisy_32'
if not (ONNX_REPORT / 'aggregate.json').is_file():
    onnx_command = [sys.executable, 'scripts/benchmark_asr_v2.py', str(MANIFEST), '--direction','vi2en','--split','dev','--audio','noisy','--denoiser','passthrough','--max-samples',str(SMOKE_SAMPLES),'--progress-every','8','--config',str(CONFIG),'--report-dir',str(ONNX_REPORT),'--resume']
    run_streaming(onnx_command, 'GIPFormer ONNX baseline')
else: print('ONNX baseline already complete; skipping.')

PYTORCH_REPORT = REPORT_ROOT / 'pytorch_dev_noisy_32'
command = [sys.executable, 'scripts/benchmark_gipformer_pytorch.py', str(MANIFEST), '--infer-script',str(UPSTREAM_DIR/'infer_pytorch.py'),'--python',str(UPSTREAM_PYTHON),'--model-dir',str(MODEL_ROOT),'--icefall-dir',str(ICEFALL_ROOT),'--device','cuda','--split','dev','--audio','noisy','--max-samples',str(SMOKE_SAMPLES),'--batch-size','8','--resume','--baseline-aggregate',str(ONNX_REPORT/'aggregate.json'),'--max-regression-pp','1.0','--report-dir',str(PYTORCH_REPORT)]
run_streaming(command, 'GIPFormer PyTorch compatibility')


In [ ]:
result = json.loads((PYTORCH_REPORT / 'aggregate.json').read_text(encoding='utf-8'))
print(json.dumps(result, ensure_ascii=False, indent=2))
if not result['compatibility_gate']['passed']: raise RuntimeError('Gate is not approved; keep the current ONNX runtime.')
decision = {'status':'PYTORCH_CHECKPOINT_COMPATIBLE','training_started':False,'next_required_work':'Obtain or build a reviewed Icefall Vietnamese construction training recipe. Official upstream currently supplies PyTorch inference only.','report':str(PYTORCH_REPORT/'aggregate.json')}
(WORK_ROOT/'compatibility_decision.json').write_text(json.dumps(decision, ensure_ascii=False, indent=2), encoding='utf-8')
print('PASS: official checkpoint matches the ONNX baseline within 1 percentage point.')
print('Fine-tuning intentionally has not started; the next task is a reviewed train/dev recipe, never an improvised command.')
